In [1]:
import re
import pandas as pd

# 1. Load your dataset
# Replace 'your_labeled_data.csv' with your actual filename
df = pd.read_csv('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/train_labeled.csv')

def audit_confidences(text):
    # Ensure text is a string
    text = str(text)
    
    # Extract specific confidence patterns
    label_conf = re.search(r"Label_Confidence:\s*([0-9.]+)", text, re.IGNORECASE)
    reason_conf = re.search(r"Reasoning_Confidence:\s*([0-9.]+)", text, re.IGNORECASE)
    
    # Helper to safely convert to float
    def safe_float(match):
        if match:
            try:
                val = float(match.group(1))
                return val if val <= 1.0 else val / 100
            except ValueError:
                return None
        return None

    return {
        "has_label_conf": label_conf is not None,
        "has_reason_conf": reason_conf is not None,
        "label_val": safe_float(label_conf),
        "reason_val": safe_float(reason_conf)
    }

# 2. Apply the audit to the 'text' column
# This expands the dictionary into 4 new columns
audit_results = df['text'].apply(audit_confidences).apply(pd.Series)

# 3. Combine with original data
final_df = pd.concat([df, audit_results], axis=1)

# 4. Generate Audit Statistics for your Thesis
print("--- Audit Statistics ---")
print(f"Total Tickets: {len(final_df)}")
print(f"Tickets with Label_Confidence: {final_df['has_label_conf'].sum()}")
print(f"Tickets with Reasoning_Confidence: {final_df['has_reason_conf'].sum()}")

# Calculate correlation if both exist
both_exist = final_df.dropna(subset=['label_val', 'reason_val'])
if not both_exist.empty:
    correlation = both_exist['label_val'].corr(both_exist['reason_val'])
    print(f"Correlation between Label and Reasoning scores: {correlation:.4f}")
    print(f"Average Difference: {(both_exist['label_val'] - both_exist['reason_val']).abs().mean():.4f}")

# 5. Save the audited file
#final_df.to_csv('audited_silver_standard.csv', index=False)

--- Audit Statistics ---
Total Tickets: 18461
Tickets with Label_Confidence: 17776
Tickets with Reasoning_Confidence: 17769
Correlation between Label and Reasoning scores: 0.8907
Average Difference: 0.0285


In [7]:
import re
import pandas as pd
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
INPUT_CSV_PATH = '/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/train/train.csv'
OUTPUT_CSV_PATH = '/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/alsei/Thesis_IT_TicketClassification/data/nuuday_distillation_18k.csv'

# ==========================================
# 2. UPDATED PARSING LOGIC (FIXED REDUNDANCY)
# ==========================================
def parse_teacher_output(row_text):
    row_text = str(row_text)
    
    # 1. Extract Description
    desc_match = re.search(r"description:\s*(.*?)\s*(?:Please clean up|Category:|Tag:)", row_text, re.DOTALL | re.IGNORECASE)
    description = desc_match.group(1).strip() if desc_match else "N/A"
    
    # 2. Extract Tag: We look specifically for 'Tag:' followed by the (Category, Sub), Scenario
    # We stop as soon as we see the word 'Reasoning:' (case insensitive)
    tag_match = re.search(r"Tag:\s*(.*?)\s*(?:Reasoning:|$)", row_text, re.DOTALL | re.IGNORECASE)
    label = tag_match.group(1).strip() if tag_match else "UNKNOWN"
    
    # 3. Extract Reasoning: We want the logic that comes AFTER the tag.
    # The Teacher output often has "reasoning: Tag: ... Reasoning: [THIS IS WHAT WE WANT]"
    # We use re.findall to find all occurrences and take the LAST one, which is usually the actual logic.
    reasoning_blocks = re.findall(r"Reasoning:\s*(.*?)\s*(?:Reasoning_Confidence:|$)", row_text, re.DOTALL | re.IGNORECASE)
    reasoning = reasoning_blocks[-1].strip() if reasoning_blocks else "N/A"
    
    # 4. Extract Scientific Confidence (S*)
    s_star_match = re.search(r"scientific_confidence:\s*([0-9.]+)", row_text, re.IGNORECASE)
    s_star = float(s_star_match.group(1)) if s_star_match else 0.0
    
    return description, label, reasoning, s_star

# ==========================================
# 3. EXECUTION
# ==========================================
def main():
    print(f"Loading raw data from: {INPUT_CSV_PATH}")
    df = pd.read_csv(INPUT_CSV_PATH)

    print("Parsing components and fixing reasoning redundancy...")
    df[['clean_desc', 'clean_label', 'clean_reasoning', 's_star']] = df['text'].apply(
        lambda x: pd.Series(parse_teacher_output(x))
    )

    # We format the 'label' column to be a clean, structured target for the Student LLM
    def create_composite_label(row):
        return (
            f"Reasoning: {row['clean_reasoning']}\n"
            f"Tag: {row['clean_label']}\n"
            f"Scientific Confidence: {row['s_star']}"
        )

    print("Creating minimalist dataset for train.py...")
    final_df = pd.DataFrame({
        'text': df['clean_desc'],
        'label': df.apply(create_composite_label, axis=1),
        's_star': df['s_star']
    })

    final_df.to_csv(OUTPUT_CSV_PATH, index=False)
    
    print("-" * 30)
    print(f"SUCCESS: {len(final_df)} rows saved.")
    print("-" * 30)
    print("VERIFICATION OF TARGET (LABEL):")
    print(final_df['label'].iloc[0]) # Should now be clean
    print("-" * 30)

if __name__ == "__main__":
    main()

Loading raw data from: /afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/train/train.csv
Parsing components and fixing reasoning redundancy...
Creating minimalist dataset for train.py...
------------------------------
SUCCESS: 100 rows saved.
------------------------------
VERIFICATION OF TARGET (LABEL):
Reasoning: Tag: (1a MV Mobile voice, 2a Nummerportering), 3 Port in
Reasoning: The description indicates a mobile number that hasn't activated in Dawn with open flows in porting-related systems (OCH, Nabs), and "NP/SP" suggests number porting. This fits mobile voice number porting, most likely an incoming port (Port in) that needs cleanup to proceed.
Tag: (1a MV Mobile voice, 2a Nummerportering), 3 Port in
Scientific Confidence: 0.788
------------------------------


In [4]:
import os
print("Your file is located here:")
print(os.path.abspath('nuuday_distillation_full_18k.csv'))

Your file is located here:
/mnt/batch/tasks/shared/LS_root/mounts/clusters/alextest/code/nuuday_distillation_full_18k.csv
